# Session 2.2 -- Chunking and Vector Ingestion

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Compare** fixed-size, overlap, and paragraph-aware chunking strategies and select one appropriate for a given document type
2. **Explore** the pre-built chunking function, experiment with parameters that split documents and attach structured metadata
3. **Ingest** chunked and embedded documents into ChromaDB and verify retrieval with a similarity query

## What You Are Working With

- `src/s3_ingestion/chunker.py` -- Chunking module with `Chunk` dataclass and `chunk_document()` / `chunk_fixed()` (provided complete)
- `src/s3_ingestion/store.py` -- ChromaDB storage with `ingest_chunks()`, `verify_ingestion()`, `reset_collection()` (provided complete)
- `src/s2_embeddings/embed.py` -- Voyage AI embedding functions from Session 2.1 (provided complete)
- `data/northbrook/` -- Corpus of memos, meeting notes, financial reports, and policies from Northbrook Partners

You **import and use** the pre-built modules. You do not need to build them from scratch.

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
from pathlib import Path
import textwrap

# Pre-built chunking module
from src.s3_ingestion.chunker import chunk_document, chunk_fixed, Chunk

# Pre-built storage module
from src.s3_ingestion.store import get_collection, ingest_chunks, verify_ingestion, reset_collection

# Embedding module from Session 2.1
from src.s2_embeddings.embed import embed_texts, cosine_similarity

print("All imports loaded successfully.")

In [ ]:
# Reset the collection so this notebook is idempotent.
# Running this cell guarantees a clean ChromaDB state every time.
reset_collection()
print("Collection reset. Starting fresh.")

---

## 2. Why Chunking Matters

In Session 2.1, we embedded short text snippets and measured similarity. In the real world, documents are long -- a policy might be 3,000 characters, a handbook might be 5,000. You cannot embed the whole thing as one vector and expect good retrieval.

Let's see the problem firsthand.

In [ ]:
# Load a full Northbrook document
doc_path = Path("../../data/northbrook/employee_handbook.md")
handbook_text = doc_path.read_text()

print(f"Document: {doc_path.name}")
print(f"Length:   {len(handbook_text):,} characters")
print(f"Lines:    {len(handbook_text.splitlines())}")
print()
print("--- First 300 characters ---")
print(handbook_text[:300])
print("...")

In [ ]:
# This document covers many different topics. Let's see what sections it contains.
sections = [line for line in handbook_text.splitlines() if line.startswith("## ")]
print(f"Sections in the handbook ({len(sections)} total):")
for s in sections:
    print(f"  {s}")

In [ ]:
# The problem: if we embed this entire document as ONE vector, that single
# vector has to represent Dress Code AND Performance Reviews AND Onboarding
# AND Professional Development all at once.
#
# Let's prove it. Embed the whole document, then compare it to specific questions.

full_doc_embedding = embed_texts([handbook_text])[0]

questions = [
    "What is the dress code at Northbrook Partners?",
    "How does the performance review process work?",
    "What is the professional development budget?",
    "What is the Navigator onboarding program?",
]

question_embeddings = embed_texts(questions)

print("Similarity between FULL DOCUMENT embedding and each question:\n")
for q, q_emb in zip(questions, question_embeddings):
    sim = cosine_similarity(full_doc_embedding, q_emb)
    print(f"  {sim:.4f}  {q}")

print("\nNotice: all scores are moderate. The single embedding tries to represent")
print("everything and ends up representing nothing well.")

**The tradeoff at the heart of chunking:**

- **Too big** = diluted meaning. A single vector cannot represent five different topics well.
- **Too small** = lost context. A sentence fragment like "the policy applies" means nothing on its own.

Every chunking strategy is a different answer to this tension.

---

## 3. Exploring `chunk_document()`

The pre-built module `src/s3_ingestion/chunker.py` provides two functions:

- **`chunk_document()`** -- paragraph-aware chunking (merges small paragraphs, respects boundaries)
- **`chunk_fixed()`** -- simple fixed-size chunking (every N chars with overlap)

Both return `Chunk` objects with `.text` and `.metadata`.

Let's start by exploring `chunk_document()` with different parameters.

In [ ]:
# Load the vacation policy -- a well-structured document with clear sections
vacation_text = Path("../../data/northbrook/vacation_policy_2025.md").read_text()

print(f"Document: vacation_policy_2025.md")
print(f"Length:   {len(vacation_text):,} characters")
print()

# Chunk it with default parameters
chunks = chunk_document(vacation_text, source="vacation_policy_2025.md", doc_type="policy")

print(f"Chunks created: {len(chunks)}")
print(f"Average chunk size: {sum(len(c.text) for c in chunks) / len(chunks):.0f} characters")
print()

# Preview each chunk
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk.text)} chars) ---")
    print(chunk.text[:150].replace("\n", " "))
    print("...")
    print()

In [ ]:
# Now let's vary the chunk_size and see the impact
print(f"{'chunk_size':>12}  {'overlap':>8}  {'chunks':>8}  {'avg_size':>10}  {'min_size':>10}  {'max_size':>10}")
print("-" * 70)

for size in [200, 300, 500, 800, 1000, 1500]:
    overlap = size // 5  # 20% overlap
    test_chunks = chunk_document(
        vacation_text,
        source="vacation_policy_2025.md",
        chunk_size=size,
        overlap=overlap,
    )
    sizes = [len(c.text) for c in test_chunks]
    avg = sum(sizes) / len(sizes)
    print(f"{size:>12}  {overlap:>8}  {len(test_chunks):>8}  {avg:>10.0f}  {min(sizes):>10}  {max(sizes):>10}")

print()
print("Smaller chunk_size = more chunks = more embeddings = higher cost")
print("Larger chunk_size  = fewer chunks = less cost = but less precise retrieval")

In [ ]:
# Interactive exploration: vary chunk_size and overlap to feel the tradeoff.
# If ipywidgets is available, this creates interactive sliders.
# If not, we fall back to a static comparison.

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    def explore_chunking(chunk_size=500, overlap=100):
        test_chunks = chunk_document(
            vacation_text,
            source="vacation_policy_2025.md",
            chunk_size=chunk_size,
            overlap=overlap,
        )
        sizes = [len(c.text) for c in test_chunks]
        avg_size = sum(sizes) / len(sizes) if sizes else 0

        # Estimated embedding cost (Voyage AI voyage-3-lite: ~$0.00006 per 1K chars)
        total_chars = sum(sizes)
        est_cost = total_chars / 1000 * 0.00006

        print(f"chunk_size={chunk_size}, overlap={overlap} ({overlap/chunk_size*100:.0f}%)")
        print(f"  Chunks: {len(test_chunks)}")
        print(f"  Avg size: {avg_size:.0f} chars | Min: {min(sizes)} | Max: {max(sizes)}")
        print(f"  Total characters: {total_chars:,}")
        print(f"  Est. embedding cost: ${est_cost:.6f}")

        # Filmstrip: show first 5 chunks with boundaries
        print(f"\n  --- First {min(5, len(test_chunks))} chunks (filmstrip) ---")
        for i, chunk in enumerate(test_chunks[:5]):
            preview = chunk.text[:80].replace("\n", " ")
            print(f"  [{i}] ({len(chunk.text):>4} chars) {preview}...")

    size_slider = widgets.IntSlider(value=500, min=200, max=1200, step=50, description="chunk_size")
    overlap_slider = widgets.IntSlider(value=100, min=0, max=300, step=25, description="overlap")

    widgets.interact(explore_chunking, chunk_size=size_slider, overlap=overlap_slider)

except ImportError:
    print("ipywidgets not available -- showing static comparison instead.")
    print("Install with: pip install ipywidgets\n")

    for size, overlap in [(200, 0), (500, 100), (1000, 200)]:
        test_chunks = chunk_document(
            vacation_text,
            source="vacation_policy_2025.md",
            chunk_size=size,
            overlap=overlap,
        )
        sizes = [len(c.text) for c in test_chunks]
        print(f"chunk_size={size}, overlap={overlap}: {len(test_chunks)} chunks, avg {sum(sizes)//len(sizes)} chars")
        for i, c in enumerate(test_chunks[:3]):
            print(f"  [{i}] {c.text[:60].replace(chr(10), ' ')}...")
        print()

---

## 4. Chunk Destruction Demo

What happens when chunking splits a critical detail across two chunks? The answer straddles the boundary and **neither chunk** contains the full information.

This is the seam problem.

In [ ]:
# Take a specific paragraph from the vacation policy that contains a complete thought
# with conditions: the Recharge Days section.
# This paragraph contains: what it is, how many, how to use, and the carryover rule.

recharge_section = """Northbrook Partners is introducing Recharge Days as a new benefit for 2025. In addition to standard PTO, all employees receive 2 Recharge Days per calendar year. Recharge Days are company-encouraged mental health days that require no justification or advance notice. Employees simply notify their manager that they are taking a Recharge Day. Recharge Days do not carry over and must be used within the calendar year. The intent of Recharge Days is to normalize proactive rest. We encourage all employees to take advantage of this benefit."""

print("=== FULL PARAGRAPH (coherent) ===")
print()
print(textwrap.fill(recharge_section, width=80))
print(f"\nLength: {len(recharge_section)} characters")

In [ ]:
# Now chunk it with fixed-size, NO overlap -- watch the destruction
fixed_no_overlap = chunk_fixed(
    recharge_section,
    source="demo",
    chunk_size=250,
    overlap=0,
)

print(f"Fixed chunking (250 chars, NO overlap) --> {len(fixed_no_overlap)} chunks\n")

for i, chunk in enumerate(fixed_no_overlap):
    print(f"--- Chunk {i} ({len(chunk.text)} chars) ---")
    print(chunk.text)
    if i < len(fixed_no_overlap) - 1:
        # Show the cut point
        end_chars = chunk.text[-30:]
        next_start = fixed_no_overlap[i + 1].text[:30]
        print(f"\n  >>> CUT POINT <<<")
        print(f"  Chunk {i} ends with:    ...{end_chars}")
        print(f"  Chunk {i+1} starts with: {next_start}...")
    print()

In [ ]:
# Now with overlap -- see how it recovers some context at the seam
fixed_with_overlap = chunk_fixed(
    recharge_section,
    source="demo",
    chunk_size=250,
    overlap=50,
)

print(f"Fixed chunking (250 chars, 50 overlap) --> {len(fixed_with_overlap)} chunks\n")

for i, chunk in enumerate(fixed_with_overlap):
    print(f"--- Chunk {i} ({len(chunk.text)} chars) ---")
    print(chunk.text)
    print()

print("Overlap recovers some context at the boundaries, but it is not free --")
print(f"we went from {len(fixed_no_overlap)} chunks to {len(fixed_with_overlap)} chunks.")
print("More chunks = more embeddings = more cost.")

In [ ]:
# Compare: paragraph-aware chunking on the same text keeps it whole
para_chunks = chunk_document(
    recharge_section,
    source="demo",
    chunk_size=600,
)

print(f"Paragraph-aware chunking (600 chars) --> {len(para_chunks)} chunk(s)\n")
for i, chunk in enumerate(para_chunks):
    print(f"--- Chunk {i} ({len(chunk.text)} chars) ---")
    print(textwrap.fill(chunk.text, width=80))
    print()

print("The paragraph-aware strategy kept the complete thought together.")
print("No cut, no lost context, no dangling fragments.")

**Takeaway:** The seam problem is real. When chunking splits a critical detail, neither chunk tells the full story. Overlap helps, but paragraph-aware chunking avoids the problem entirely for well-structured documents.

---

## 5. Chunk Autopsy

Every `Chunk` object has two fields:
- `.text` -- the chunk content
- `.metadata` -- a dictionary with information about where this chunk came from

Metadata is what lets us filter and trace results back to their source. Let's inspect a real chunk in detail.

In [ ]:
# Chunk the vacation policy and inspect a single chunk in detail
chunks = chunk_document(
    vacation_text,
    source="vacation_policy_2025.md",
    doc_type="policy",
)

# Pick chunk 2 (index 2) as our specimen
specimen = chunks[2]

print("=== Chunk Autopsy ===")
print(f"\nChunk index: 2 of {len(chunks)} total")
print(f"Text length: {len(specimen.text)} characters")
print()
print("--- Metadata ---")
for key, value in specimen.metadata.items():
    print(f"  {key:15s} = {value!r}")
print()
print("--- Full Text ---")
print(specimen.text)

In [ ]:
# Metadata is what ChromaDB stores alongside the embedding.
# When you retrieve a chunk, metadata tells you:
#   - source: which document it came from
#   - chunk_index: where in the document it appeared
#   - doc_type: the category of document
#
# In Week 3, metadata will enable FILTERED retrieval:
#   "Find chunks about vacation, but ONLY from policy documents"

print("Metadata for all chunks in this document:\n")
print(f"{'Index':>6}  {'Source':<30}  {'doc_type':<10}  {'chunk_index':<12}  {'Length':>6}")
print("-" * 75)
for chunk in chunks:
    m = chunk.metadata
    print(f"{m.get('chunk_index', '?'):>6}  {m.get('source', '?'):<30}  {m.get('doc_type', '?'):<10}  {m.get('chunk_index', '?'):<12}  {len(chunk.text):>6}")

In [ ]:
# Chunk Boundary Inspection
# For each pair of adjacent chunks, show the boundary.
# Good boundaries end at a natural break (end of paragraph, end of sentence).
# Bad boundaries cut mid-sentence or mid-word.

print("=== Chunk Boundaries ===\n")

for i in range(len(chunks) - 1):
    end_text = chunks[i].text[-60:].replace("\n", " ")
    start_text = chunks[i + 1].text[:60].replace("\n", " ")

    # Simple heuristic: does the boundary land at a sentence end?
    ends_clean = end_text.rstrip().endswith(('.', '!', '?', ':'))
    quality = "natural break" if ends_clean else "mid-text"

    print(f"Boundary between chunk {i} and chunk {i+1}  [{quality}]")
    print(f"  ...{end_text}")
    print(f"  {start_text}...")
    print()

**Build this habit:** Always inspect your chunk boundaries. Do not assume the chunker produces good output. The quality of your chunks directly determines the quality of your retrieval.

---

## 6. Fixed vs Paragraph-Aware Chunking

The module provides two strategies:
- **`chunk_fixed()`** -- cuts every N characters, ignoring content boundaries
- **`chunk_document()`** -- merges paragraphs up to a size limit, respects natural breaks

Let's run both on the same document and compare.

In [ ]:
# Load the remote work policy -- another structured document
remote_text = Path("../../data/northbrook/remote_work_policy.md").read_text()

# Fixed-size chunking
fixed_chunks = chunk_fixed(
    remote_text,
    source="remote_work_policy.md",
    chunk_size=500,
    overlap=100,
    doc_type="policy",
)

# Paragraph-aware chunking
para_chunks = chunk_document(
    remote_text,
    source="remote_work_policy.md",
    chunk_size=500,
    overlap=100,
    doc_type="policy",
)

print(f"{'':25s}  {'Fixed':>10}  {'Paragraph':>12}")
print("-" * 50)

fixed_sizes = [len(c.text) for c in fixed_chunks]
para_sizes = [len(c.text) for c in para_chunks]

print(f"{'Chunk count':25s}  {len(fixed_chunks):>10}  {len(para_chunks):>12}")
print(f"{'Average size (chars)':25s}  {sum(fixed_sizes)//len(fixed_sizes):>10}  {sum(para_sizes)//len(para_sizes):>12}")
print(f"{'Min size':25s}  {min(fixed_sizes):>10}  {min(para_sizes):>12}")
print(f"{'Max size':25s}  {max(fixed_sizes):>10}  {max(para_sizes):>12}")
print(f"{'Total characters':25s}  {sum(fixed_sizes):>10}  {sum(para_sizes):>12}")

In [ ]:
# Side-by-side: first 3 chunks from each strategy
print("=" * 80)
print("FIXED-SIZE CHUNKS")
print("=" * 80)
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\n--- Fixed Chunk {i} ({len(chunk.text)} chars) ---")
    print(textwrap.fill(chunk.text[:200], width=80))
    if len(chunk.text) > 200:
        print("...")

print()
print("=" * 80)
print("PARAGRAPH-AWARE CHUNKS")
print("=" * 80)
for i, chunk in enumerate(para_chunks[:3]):
    print(f"\n--- Paragraph Chunk {i} ({len(chunk.text)} chars) ---")
    print(textwrap.fill(chunk.text[:200], width=80))
    if len(chunk.text) > 200:
        print("...")

print()
print("Notice the difference: fixed chunks cut wherever the character count lands.")
print("Paragraph chunks respect natural boundaries between topics.")

In [ ]:
# Boundary quality comparison
def count_clean_boundaries(chunks_list):
    """Count how many chunk boundaries end at a natural break."""
    clean = 0
    total = len(chunks_list) - 1
    for i in range(total):
        if chunks_list[i].text.rstrip().endswith(('.', '!', '?', ':', '\n')):
            clean += 1
    return clean, total

fixed_clean, fixed_total = count_clean_boundaries(fixed_chunks)
para_clean, para_total = count_clean_boundaries(para_chunks)

print("Boundary Quality:\n")
print(f"  Fixed:     {fixed_clean}/{fixed_total} boundaries end at a natural break ({fixed_clean/max(fixed_total,1)*100:.0f}%)")
print(f"  Paragraph: {para_clean}/{para_total} boundaries end at a natural break ({para_clean/max(para_total,1)*100:.0f}%)")

**Key insight:** For structured documents with clear sections and paragraphs (like the Northbrook policies), paragraph-aware chunking produces higher quality chunks. For unstructured content without paragraph breaks (like raw logs), fixed-size may be the only option.

Neither strategy is universally "best." The right choice depends on your documents.

---

## 7. Vector Store Ingestion

Now that we can chunk documents, we need to store them in a place where we can search by meaning. That place is **ChromaDB** -- an open-source vector database.

The pipeline is:

```
Document  -->  chunk_document()  -->  Chunk[]  -->  ingest_chunks()  -->  ChromaDB
              (text + metadata)                   (embed + store)
```

The `ingest_chunks()` function from `store.py` handles the embed-and-store step. Let's walk through it.

In [ ]:
# First, let's understand what ingest_chunks() does by looking at its behavior.
# It takes a list of Chunk objects and:
#   1. Extracts the text from each chunk
#   2. Calls embed_texts() (Voyage AI) in batches of 50
#   3. Generates a deterministic ID for each chunk: "{source}_chunk{index}"
#   4. Adds everything to a ChromaDB collection

# Let's ingest the vacation policy chunks
vacation_chunks = chunk_document(
    vacation_text,
    source="vacation_policy_2025.md",
    doc_type="policy",
)

print(f"Ingesting {len(vacation_chunks)} chunks from vacation_policy_2025.md...\n")

result = ingest_chunks(vacation_chunks)
print(f"\nResult: {result}")

In [ ]:
# Why deterministic IDs matter:
# The ID for each chunk is constructed from its source and index.
# This means if you run ingestion twice, ChromaDB rejects duplicates
# instead of creating copies. That is idempotency.

print("Generated IDs for the vacation policy chunks:\n")
for chunk in vacation_chunks:
    chunk_id = f"{chunk.metadata['source']}_chunk{chunk.metadata['chunk_index']}"
    print(f"  {chunk_id}")

In [ ]:
# Let's check the collection state
collection = get_collection()
print(f"Collection: {collection.name}")
print(f"Total documents stored: {collection.count()}")

In [ ]:
# Now ingest the FULL Northbrook corpus
data_dir = Path("../../data/northbrook")
all_files = sorted(data_dir.glob("*.md"))

print(f"Found {len(all_files)} documents in the Northbrook corpus:\n")
for f in all_files:
    print(f"  {f.name} ({len(f.read_text()):,} chars)")

In [ ]:
# Reset the collection so we start clean for the full corpus ingestion
reset_collection()
print("Collection reset for full corpus ingestion.\n")

# Determine doc_type from filename
def classify_doc_type(filename: str) -> str:
    """Simple classifier based on filename patterns."""
    name = filename.lower()
    if "memo" in name:
        return "memo"
    elif "policy" in name or "handbook" in name:
        return "policy"
    elif "financial" in name:
        return "financial"
    elif "meeting" in name or "standup" in name or "committee" in name:
        return "meeting"
    else:
        return "unknown"

# Chunk and ingest every document
total_chunks = 0
total_ingested = 0

for doc_path in all_files:
    text = doc_path.read_text()
    doc_type = classify_doc_type(doc_path.name)

    chunks = chunk_document(
        text,
        source=doc_path.name,
        doc_type=doc_type,
    )

    total_chunks += len(chunks)
    print(f"  {doc_path.name:<45s}  type={doc_type:<12s}  chunks={len(chunks)}")

    result = ingest_chunks(chunks)
    total_ingested += result["ingested"]

print(f"\n{'=' * 60}")
print(f"Total documents processed: {len(all_files)}")
print(f"Total chunks created:      {total_chunks}")
print(f"Total chunks ingested:     {total_ingested}")

In [ ]:
# Verify the collection is populated
collection = get_collection()
print(f"Collection '{collection.name}' now contains {collection.count()} chunks.")
print(f"\nThis collection persists to disk at ./chroma_db/")
print("It will survive between sessions -- you will use it in Week 3 for RAG.")

---

## 8. Verification

The `verify_ingestion()` function embeds a query, searches the collection, and returns the most similar chunks. This is the retrieval step that will form the foundation of RAG next week.

Let's test it.

In [ ]:
# Run a verification query
results = verify_ingestion("What is the vacation policy at Northbrook Partners?", n_results=5)

print("Query: 'What is the vacation policy at Northbrook Partners?'\n")

for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    doc_text = results["documents"][0][i]
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i]
    similarity = 1 - distance  # ChromaDB cosine distance = 1 - similarity

    print(f"[{i+1}] Score: {similarity:.4f}  |  Source: {metadata.get('source', '?')}  |  Type: {metadata.get('doc_type', '?')}")
    print(f"    Chunk index: {metadata.get('chunk_index', '?')}")
    print(f"    {doc_text[:150].replace(chr(10), ' ')}...")
    print()

In [ ]:
# Try several different queries to see how the retrieval works
test_queries = [
    "What is the remote work policy?",
    "How much is the professional development budget?",
    "What was discussed at the board meeting?",
    "What are the security update details?",
    "How does the Navigator onboarding program work?",
]

for query in test_queries:
    results = verify_ingestion(query, n_results=2)
    print(f"Q: {query}")
    for i in range(len(results["ids"][0])):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]
        similarity = 1 - distance
        text_preview = results["documents"][0][i][:80].replace("\n", " ")
        print(f"  [{similarity:.4f}] {metadata.get('source', '?'):40s} {text_preview}...")
    print()

**What to notice:**

- The scores tell you how similar the retrieved chunk is to your query (higher = more similar)
- The metadata tells you which document and section the chunk came from
- Different queries surface chunks from different documents -- the retrieval is working
- Some results may surprise you -- semantic similarity is not the same as keyword matching

---

## 9. Same Question, Different Chunks

This is the proof that chunking strategy has real downstream consequences. We will:

1. Take the same corpus
2. Chunk it two ways: paragraph-aware vs fixed-size with no overlap
3. Ingest each into a separate ChromaDB collection
4. Run the same query against both
5. Compare the results

In [ ]:
# Create two separate collections for comparison
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

# Clean up comparison collections if they exist
for name in ["compare_paragraph", "compare_fixed"]:
    try:
        client.delete_collection(name)
    except Exception:
        pass

print("Comparison collections reset.")

In [ ]:
# Chunk the entire corpus two ways and ingest into separate collections

paragraph_all_chunks = []
fixed_all_chunks = []

for doc_path in all_files:
    text = doc_path.read_text()
    doc_type = classify_doc_type(doc_path.name)

    # Paragraph-aware (default strategy)
    p_chunks = chunk_document(text, source=doc_path.name, doc_type=doc_type, chunk_size=500, overlap=100)
    paragraph_all_chunks.extend(p_chunks)

    # Fixed-size, NO overlap (worst case)
    f_chunks = chunk_fixed(text, source=doc_path.name, chunk_size=500, overlap=0, doc_type=doc_type)
    fixed_all_chunks.extend(f_chunks)

print(f"Paragraph-aware: {len(paragraph_all_chunks)} total chunks")
print(f"Fixed (no overlap): {len(fixed_all_chunks)} total chunks")

# Ingest paragraph chunks
print("\nIngesting paragraph-aware chunks...")
p_result = ingest_chunks(paragraph_all_chunks, collection_name="compare_paragraph")
print(f"  Done: {p_result}")

# Ingest fixed chunks
print("\nIngesting fixed-size chunks...")
f_result = ingest_chunks(fixed_all_chunks, collection_name="compare_fixed")
print(f"  Done: {f_result}")

In [ ]:
# Now run the SAME query against BOTH collections
comparison_query = "What is the vacation policy at Northbrook Partners?"

para_results = verify_ingestion(comparison_query, n_results=3, collection_name="compare_paragraph")
fixed_results = verify_ingestion(comparison_query, n_results=3, collection_name="compare_fixed")

print(f"Query: '{comparison_query}'")
print()

print("=" * 80)
print("PARAGRAPH-AWARE RESULTS")
print("=" * 80)
for i in range(len(para_results["ids"][0])):
    dist = para_results["distances"][0][i]
    meta = para_results["metadatas"][0][i]
    text = para_results["documents"][0][i]
    print(f"\n[{i+1}] Score: {1-dist:.4f} | Source: {meta.get('source', '?')}")
    print(f"    {textwrap.fill(text[:200], width=76, initial_indent='    ', subsequent_indent='    ')}")
    if len(text) > 200:
        print("    ...")

print()
print("=" * 80)
print("FIXED-SIZE (NO OVERLAP) RESULTS")
print("=" * 80)
for i in range(len(fixed_results["ids"][0])):
    dist = fixed_results["distances"][0][i]
    meta = fixed_results["metadatas"][0][i]
    text = fixed_results["documents"][0][i]
    print(f"\n[{i+1}] Score: {1-dist:.4f} | Source: {meta.get('source', '?')}")
    print(f"    {textwrap.fill(text[:200], width=76, initial_indent='    ', subsequent_indent='    ')}")
    if len(text) > 200:
        print("    ...")

In [ ]:
# Score comparison summary
print("Score Comparison (higher = better):\n")

para_scores = [1 - d for d in para_results["distances"][0]]
fixed_scores = [1 - d for d in fixed_results["distances"][0]]

print(f"{'Rank':>5}  {'Paragraph':>12}  {'Fixed':>12}  {'Difference':>12}")
print("-" * 45)
for i in range(min(len(para_scores), len(fixed_scores))):
    diff = para_scores[i] - fixed_scores[i]
    marker = " <--" if abs(diff) > 0.01 else ""
    print(f"{i+1:>5}  {para_scores[i]:>12.4f}  {fixed_scores[i]:>12.4f}  {diff:>+12.4f}{marker}")

print()
print("The chunking strategy you choose affects retrieval quality.")
print("The model downstream has no way to recover from bad chunks.")

In [ ]:
# Clean up comparison collections -- we keep the main 'northbrook' collection
try:
    client.delete_collection("compare_paragraph")
    client.delete_collection("compare_fixed")
    print("Comparison collections cleaned up.")
    print(f"Main collection 'northbrook' still has {get_collection().count()} chunks.")
except Exception as e:
    print(f"Cleanup note: {e}")

**This is the proof.** Same corpus, same query, different chunking strategy, different results.

Paragraph-aware chunking typically returns more complete, coherent chunks that score higher on similarity. Fixed-size chunking with no overlap often returns fragments that start or end mid-sentence.

> *"If your chunks are bad, no amount of prompt engineering will save you."*

---

## 10. Bridge to RAG

Here is where we are in the pipeline:

| Session | What We Built | Status |
|---------|--------------|--------|
| **1.1** | API connection + generation | Done |
| **1.2** | Structured extraction with Pydantic | Done |
| **2.1** | Embeddings + similarity measurement | Done |
| **2.2** | Chunking + vector store ingestion | **Done (today)** |
| **3.1** | Naive RAG -- retrieve + generate | Next session |
| **3.2** | Metadata-filtered RAG + evaluation | Coming |

We now have:
- Structured data extraction (Session 1.2)
- Embeddings that capture meaning (Session 2.1)
- A searchable vector store populated with the Northbrook corpus (today)

**Next session:** Retrieval Augmented Generation. We take a question, retrieve the most relevant chunks from ChromaDB, and hand them to Claude as context for generating an answer. You will see exactly why chunking and metadata decisions matter -- because bad retrieval makes bad answers, and no prompt can fix that.

### Questions to think about before Session 3.1:

1. **On Retrieval:** If semantic search returns the "most similar" chunks, why might those not be the "most useful" chunks?
2. **On Generation:** What happens when Claude answers a question but the retrieved chunks don't actually contain the answer?
3. **On Trust:** How do you verify that an AI-generated answer actually came from your sources?

---

## 11. Session Recap

### What we built today

1. **Chunking exploration** -- compared fixed-size vs paragraph-aware strategies and felt the tradeoffs through interactive experimentation
2. **Chunk destruction demo** -- saw firsthand how bad chunking destroys meaning at boundaries
3. **Chunk autopsy** -- inspected individual chunks, their metadata, and their boundary quality
4. **Vector store ingestion** -- embedded and stored the full Northbrook corpus in ChromaDB
5. **Verification queries** -- confirmed retrieval works with similarity scores and source attribution
6. **Same question, different chunks** -- proved that chunking strategy directly impacts retrieval quality

### Key takeaways

- **Chunking is not just splitting text.** It is a design decision that directly impacts downstream retrieval quality.
- **Overlap is insurance, not a solution.** It recovers some context at seams but costs more chunks and embeddings.
- **Paragraph-aware chunking** works well for structured documents. Fixed-size is a fallback for unstructured content.
- **Metadata is cheap and powerful.** It enables filtered retrieval in Week 3.
- **ChromaDB stores chunks + embeddings + metadata** and returns the most similar results on query.
- **Deterministic IDs** make ingestion idempotent -- running it twice does not create duplicates.

### Lab 1 -- Assigned Today

Lab 1 (Batch Extraction Pipeline) brings together Weeks 1 and 2:

| Component | Source |
|-----------|--------|
| Pydantic schema design | Session 1.2 |
| Batch extraction with error handling | Session 1.2 |
| Chunking with metadata | Session 2.2 (today) |
| Output ready for ingestion | Session 2.2 (today) |

**Due:** End of Session 3.2 (two weeks) | **Weight:** 20% of course grade

Start with the schema. Build incrementally. Read the rubric before you start.